<h1>Batch Update</h1>

<p>
This notebook explains how to update a batch of records in the PIDINST site.
</p>

<h2>Template file</h2>

<p>
The same Excel template used for creating new records can also be used for batch updates.
When processing each row, the update workflow identifies the target record as follows:
</p>

<ol>
  <li>
    If <b>PKG_ID</b> is provided in the template, that record is used directly.
  </li>
  <li>
    If <b>PKG_ID</b> is not provided, the database is searched using the tuple
    <b>(Manufacturer, Model, AlternateIdentifier)</b>.
    Only the <b>first item</b> from each of these fields is considered in the search.
  </li>
  <li>
    If no matching record is found, a <b>Not Found</b> error will be reported.
  </li>
  <li>
    If a matching record is found, its metadata will be updated based on the values provided in the template.
  </li>
</ol>

<div style="border:1px solid #f5c2c7; background:#fff3cd; padding:12px 14px; border-radius:6px; margin:12px 0;">
  <h3 style="color:#E1712B"><b>Important notes </b></h3>
<ol>
  <li>
    If you want to update any of <b>Manufacturer</b>, <b>Model</b>, or <b>AlternateIdentifier</b>,
  you should provide <b>PKG_ID</b> in the template. Otherwise, the record may not be identified correctly.
  </li><br>
  <li>
    Batch update only modifies <b>metadata</b>. It does <b>not</b> update attachment resources.
  </li><br>
  <li>
    The update function can convert <b>private</b> records to <b>public</b>, but it does not convert
  public records back to private. For public records, you need to follow the UI procedure to either mark a record as <b>duplicate</b> or    <b>withdraw</b> it.
  <ul style="margin-top:8px;">
    <li><b>convert_to_public = True</b>: matching records are updated and forced to be public.</li>
    <li><b>convert_to_public = False</b>: the existing visibility is preserved. Public records remain public, and private records remain private.</li>
  </ul>
  </li>
</ol>
  
</div>

<div style="border:1px solid #b6d4fe; background:#e7f1ff; padding:12px 14px; border-radius:6px; margin:12px 0;">
    <h3 style="color:#1C69A8"><b>Note </b></h3>
<p>
If your template is already prepared, you can jump directly to <b>Step 3</b> in the notebook.
Otherwise, follow <b>Step 1</b> and <b>Step 2</b> to search for and export the desired records into the Excel template.
Then update the template as needed and continue with <b>Step 3</b>.
</p>
</div>


<div style="border:1px solid #b6d4fe; background:#e7f1ff; padding:12px 14px; border-radius:6px; margin:12px 0;">
    <h3 style="color:#1C69A8"><b>Requirements</b></h3>
  <p style="margin-top:0;">
    To run this notebook, you need the following environment variables:
    <b>CKAN_BASE_URL</b> and <b>CKAN_API_KEY</b>.
  </p>

  <p>
    You may set <b>CKAN_BASE_URL</b> directly in the notebook if needed. However, it is recommended to store
    <b>CKAN_API_KEY</b> in a <code>.env</code> file located in the same directory as this notebook.
  </p>

  <p><b>Example <code>.env</code> file:</b></p>

  <pre style="background:#f8f9fa; padding:10px; border-radius:4px; overflow-x:auto; margin:8px 0;"><code>CKAN_BASE_URL=https://pidinst-dev.data.auscope.org.au/
CKAN_API_KEY=&lt;API-KEY from ckan-site&gt;</code></pre>

  <ul style="margin-bottom:0;">
    <li><b>CKAN_BASE_URL</b>: the base URL of the PIDINST website.</li>
    <li><b>CKAN_API_KEY</b>: your API key from the CKAN site. You can create one from your user dashboard.</li>
  </ul>

  <p style="margin-bottom:0; margin-top:10px;">
    Keep your API key secure and rotate it periodically.
  </p>
</div>

In [1]:
import sys
from pprint import pprint
import os
from dotenv import load_dotenv

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s",
    stream=sys.stdout,
    force=True,  # important in notebooks
)

from ckan_batch.client import CKANClient
from ckan_batch.reader.pidinst import read_pidinst_template

In [2]:
load_dotenv()

CKAN_BASE_URL = os.getenv("CKAN_BASE_URL", "").rstrip("/")
CKAN_API_KEY = os.getenv("CKAN_API_KEY")  # required for Create/Update/Delete (CUD) actions

# INITIALISE CLIENT

In [3]:
ckan_client = CKANClient(CKAN_BASE_URL, apikey=CKAN_API_KEY)

# Step 1 - Search and find your packages

### Generic Search

In [4]:
pkgs = ckan_client.get_all(q='title:"Phoenix Geophysics MTU-5C Magnetotelluric Data Acquisition System"',verbose=False)
len(pkgs)

9

In [11]:
pkgs = ckan_client.get_all(q='manufacturer_name_search:"Phoenix Geophysics"',verbose=True)
len(pkgs)

9

In [12]:
for pkg in pkgs:
    print(pkg.get('instrument_type')) #, {}).get('instrument_type_name')

[{"instrument_type_name": "Magnetotelluric magnetic field sensor", "instrument_type_identifier": "https://pidinst-dev.data.auscope.org.au/taxonomies/term/7a1b44d5-f474-4d3f-8363-8a7bb46cab28", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magnetotelluric data acquisition system", "instrument_type_identifier": "", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magnetotelluric data acquisition system", "instrument_type_identifier": "", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magnetotelluric data acquisition system", "instrument_type_identifier": "", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magnetotelluric data acquisition system", "instrument_type_identifier": "", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magnetotelluric data acquisition system", "instrument_type_identifier": "", "instrument_type_identifier_type": "URL"}]
[{"instrument_type_name": "Magneto

In [12]:
# pkgs = ckan_client.get_all(q="adelaide",verbose=True)
# pkgs = ckan_client.get_all(q="example",verbose=True)
# for pkg in pkgs:
#     print(pkg.get('title'))

### Search based on tuple of (manufacturer,model,serial-number)

This tuple should return a unique record. Possible returns are:

- (result_dict, None)         if exactly one match is found
- (None, [summaries])         if multiple matches are found
- (None, None)                if no match is found

In [5]:
pkg, msg = ckan_client.find_instrument_by_attributes(
    manufacturer='Phoenix Geophysics',
    model= 'MTU-5C',
    alternate_identifier= '10246',
    visibility='all',
    verbose=True
)
print(pkg.get('title'))
# Convert to list
pkgs = [pkg]

'Phoenix Geophysics MTU-5C Magnetotelluric Data Acquisition System'

# Step 2. Export to Excel for modification

In [13]:
update_path = r'C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\data\update_data\update_phoenix_geo.xlsx'

In [7]:
ckan_client.export_records(
    pkg_ids=[pkg.get('id') for pkg in pkgs],
    export_format='Excel',
    output_path=update_path
)

INFO:ckan_batch.client:Exported 9 packages, 0 not found


C:\Drive_D\projects\avre\apps\PIDINST\AuScope-instrument-registry\ckan_batch\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


INFO:ckan_batch.client:Exported data saved to C:\Users\mot032\CSIRO\AuScope AVRE - Persistent ID for Instruments (PIDINST)\data\update_data\update_phoenix_geo.xlsx


{'exported': ['3bc52542-ab80-467c-bbcb-05c56566fd0c',
  'c400bc46-7072-41d1-b45c-28cc417af6e4',
  '844abec3-c39d-4214-81d8-81116004e0ee',
  '585e5a2f-e333-427e-8458-2d93205adcc0',
  'f7880e02-871a-417f-bce3-0e8da8e381f6',
  'b2b29ecb-8348-4b4b-8b72-54d37e2a3237',
  '0dccbb72-ec6b-47a1-ad28-ee003d913eaf',
  '5901493c-c46f-4b6b-8c14-ed6cec4b4bba',
  '85269f9d-d6eb-49a4-9829-b50c23c0d3b1'],
 'not_found': [],
 'output_path': 'C:\\Users\\mot032\\CSIRO\\AuScope AVRE - Persistent ID for Instruments (PIDINST)\\data\\update_data\\update_phoenix_geo.xlsx'}

# Step 3. Read back the template with updated data

In [14]:
res_map = read_pidinst_template(
    excel_path= update_path,
    client=ckan_client,
    sheet_name="Instruments"
)
pkgs_dict = res_map.records
if len(res_map.errors) > 0:
    print("=============ERRORS==========\n")
    pprint(res_map.errors)

print(len(pkgs_dict))
# pprint(pkgs_dict[0])

9


# Step 4. Update Instruments

**NOTE:** For PIDINST set `record_type` always to "instrument", even for PLATFORMS. Both are based on the same schema, only a boolean `is_platform` separates them

In [15]:
res = ckan_client.update_records(
    records=pkgs_dict,
    convert_to_public=False,
    dry_run = False,
)

**Note:** use `res.failed` to check if there were any failure during update.

In [16]:
res.failed

[]